# WebSocket streaming — real-time run events

Almost every SDK call is a plain **request/response over HTTPS**: you send one request, the
server sends one response, the connection is done. One family of calls is different.
**`client.runs.stream()`** opens a **persistent, bidirectional WebSocket** to the workflow
server and keeps it open for the whole run:

```
wss://<host>/workflows/v1/workflow-runs/ws
```

Over that single socket:

- **client → server** — the SDK sends one opening frame: a JSON `WorkflowRunInput`
  (`workflow_id`, `command`, optional variables / per-turn inputs) that kicks the run off.
- **server → client** — the server then *pushes* a live stream of `RunEvent` frames as the
  run executes (node starts, assistant responses, edge traversals, …) until it finishes and
  closes the socket.

That two-way traffic on one long-lived connection is what makes it a bidirectional WebSocket
rather than a series of independent HTTP calls — you see events *as they happen* instead of
polling for a final result.

This notebook:
1. Shows the WebSocket endpoint and the opening frame the SDK sends
2. Streams a run's events live with `client.runs.stream()` (async context manager)
3. Holds a whole conversation on one stream: `send()` each turn, `stop()` the run
4. Inspects the `RunEvent` structure, the event types seen on the wire, and the close codes
5. Contrasts it with the ordinary HTTP request/response path (`client.runs.execute()`)

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import nest_asyncio

from interactly import AsyncWorkflowClient, WorkflowCommand
from interactly.configs import (
    DirectEdgeConfig,
    SayStaticMessageNodeConfig,
    StaticMessagesConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
)
from interactly.types.workflows.workflow import Workflow

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()

# A tiny two-node workflow we can stream events from (greeting -> goodbye).
_greeting = SayStaticMessageNodeConfig(
    name="Greeting",
    is_start=True,
    static_messages_config=StaticMessagesConfig(static_messages=["Hello! Streaming events now."]),
)
_goodbye = SayStaticMessageNodeConfig(
    name="Goodbye",
    static_messages_config=StaticMessagesConfig(static_messages=["All done — goodbye!"]),
)
_config = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(name="WebSocket Streaming Notebook", category="System Examples"),
    node_configs=[_greeting, _goodbye],
    edge_configs=[
        DirectEdgeConfig(
            source_node_logical_id=_greeting.logical_id,
            destination_node_logical_id=_goodbye.logical_id,
            name="greeting->goodbye",
        )
    ],
)
workflow: Workflow = await client.workflows.create_from_config(_config)
WORKFLOW_ID = workflow.id

print("Connected to", client._base_url)
print("Created workflow", WORKFLOW_ID)

## The WebSocket endpoint and the opening frame

`client.runs.stream()` derives a `wss://` URL from your base URL and connects to the
`/v1/workflow-runs/ws` endpoint. Before any events arrive, the SDK sends exactly one frame —
the run command, shaped as a `WorkflowRunInput`. After that the server pushes events, and the client sends a
frame only when you call `send()` (the next turn) or `stop()` (see section 1b).

The two cells below just *show* the URL and a representative opening frame so the two
directions are concrete; the next section actually opens the socket.

In [ ]:
import json

# The SDK flips the http(s) base URL to ws(s) and appends the run-stream path.
ws_url = client._ws_base_url() + "/v1/workflow-runs/ws"
print("WebSocket URL :", ws_url)

# The single frame the client sends first (client -> server) to start the run.
opening_frame = {
    "workflow_id": WORKFLOW_ID,
    "command": WorkflowCommand.START.value,
    "run_by": "api",
    # A default thread is seeded so a bare START is a valid run; a full turn would
    # carry its user message here under thread_to_node_inputs.
    "thread_to_node_inputs": {"0": {"node_run_inputs": []}},
}
print("Opening frame :", json.dumps(opening_frame, indent=2))

## 1. Stream a run's events live

`client.runs.stream(...)` returns an **async context manager**. Entering it (`async with`)
opens the WebSocket and sends the opening frame; iterating it (`async for`) yields each
`RunEvent` the server pushes over that *same* connection. The socket stays open for the whole
loop and closes automatically when the `async with` block exits. Leaving the block before the run ends
cancels the run.

`stream()` also accepts `dynamic_variables` / `runtime_variables`, or a full typed `run_input`
(a per-turn `WorkflowRunInput`) — all sent in that opening frame. Section 1b holds a whole
conversation on one stream.

In [ ]:
events_received = []

async with client.runs.stream(
    workflow_id=WORKFLOW_ID,
    command=WorkflowCommand.START,
) as stream:
    async for event in stream:
        events_received.append(event)
        text = event.output if event.output else ""
        print(f"  [{event.type:24s}] node={event.node_id or '—'}  {text}")

        if event.is_terminal():
            print(f"\n→ Terminal event ({event.type}); the server now closes the socket.")
            break

print(f"\nTotal events received over the one connection: {len(events_received)}")

## 1b. A whole conversation on one stream

One `async with` block can hold a whole conversation. When an event is `is_ready_for_input()`, send the user's
next message with `await stream.send(...)` and keep iterating the **same** stream; end the run with
`await stream.stop()`.

- `send()` sends the turn as a `resume` frame, the same frame the workflow editor sends. The variables the stream
  was opened with go with every turn.
- The server reads the socket only between turns, so a turn sent while one is running waits and is taken when
  that turn ends.
- `stop()` ends the run, which is stored `cancelled` with `termination_source="client_stop"`, not `completed`. The
  server answers "Workflow run stopped successfully" and closes the socket, so iteration ends on its own.
- Leaving the `async with` block before the run ends cancels it, even mid-turn. The exception is a durable run
  waiting on a pause of its own (a wait, an approval, a signal, a job): the server keeps it, and
  `client.runs.stream(workflow_id=..., workflow_run_id=...)` takes it back. That needs durable socket runs on the
  server.

The workflow here is one LLM node that waits for the caller and answers each message; the cell spends two short
`gpt-5.4-nano` replies.

In [ ]:
from interactly.configs import OpenAILLMConfig, OPENAIModel, PromptConfig, SayLLMNodeConfig

desk = SayLLMNodeConfig(
    name="Front desk",
    is_start=True,
    wait_for_user_message=True,
    self_loop=True,  # answer every message, one per turn
    main_response_config=PromptConfig(prompt="You are a clinic's front desk. Reply in under 15 words."),
    llms_config=OpenAILLMConfig(model=OPENAIModel.GPT_5_4_NANO, max_tokens=8192),
)
chat_workflow: Workflow = await client.workflows.create_from_config(
    WorkflowConfigFullyHydrated(
        workflow_config=WorkflowConfig(name="WebSocket conversation (15_websocket_streaming_events)", category="System Examples"),
        node_configs=[desk],
        edge_configs=[],
    )
)
CHAT_WORKFLOW_ID = chat_workflow.id

caller_lines = iter(["I need to move my appointment.", "Tuesday morning works."])

async with client.runs.stream(workflow_id=CHAT_WORKFLOW_ID) as stream:
    async for event in stream:
        if event.type == "assistant_response":
            print(f"  Desk  : {event.output}")
        if event.is_ready_for_input():
            line = next(caller_lines, None)
            if line is None:
                await stream.stop()  # ends the run; iteration ends when the server closes
            else:
                print(f"  Caller: {line}")
                await stream.send(line)
    CHAT_RUN_ID = stream.run_id

chat_run = await client.runs.get(CHAT_RUN_ID)
print(f"\nrun {CHAT_RUN_ID}: status={chat_run.status}  termination_source={chat_run.termination_source}")

## 2. The `RunEvent` structure

Each frame is parsed into a `RunEvent` with a `type`, an optional `node_id`, an `output`
payload (e.g. assistant text), a run-level `status`, and (on failure) an `error`. Unknown
event types are accepted as-is, so new server event types never break the SDK. Use
`is_terminal()`, which is true on `end_workflow` and `workflow_error`.

A `workflow_error` does not always end the run. On the run socket, a node failure the server tolerates ends
only the turn, and the user can send the next one; only a failure followed by the socket closing ends the run.
So a conversation loop should treat `workflow_error` as the end of a turn, and the run as over when iteration
stops.

In [ ]:
# Show a couple of representative frames: the first, and the terminal one.
first = events_received[0]
print("first event")
print(f"  type       : {first.type}")
print(f"  node_id    : {first.node_id}")
print(f"  output     : {first.output}")
print(f"  status     : {first.status}")
print(f"  is_terminal: {first.is_terminal()}")

last = events_received[-1]
print("\nlast event")
print(f"  type       : {last.type}")
print(f"  is_terminal: {last.is_terminal()}")

## 3. Event types seen on the stream

The exact set depends on the workflow, but a simple say-node run emits frames like these
(the list is **not** exhaustive — the server may emit others, which pass through untouched):

| Type | Meaning |
|------|---------|
| `run_ack` | handshake acknowledgement (synthetic first frame) |
| `start_node_run` / `end_node_run` | a node began / finished executing |
| `say_static` / `assistant_response` | a message the node produced (`output` holds the text) |
| `direct_edge` / `conditional_edge` | an edge transition was taken |
| `workflow_show_state` | a snapshot of run state |
| `end_thread` / `end_workflow_iteration` | a thread / iteration finished |
| `end_workflow` | the run finished (**terminal**) |
| `workflow_error` | a node failed; ends the turn, and the run too if the socket then closes |
| `run_paused` / `run_resumed` | a person paused / resumed a durable run (`is_run_paused()`, `is_run_resumed()`) |

In [ ]:
from collections import Counter

counts = Counter(e.type for e in events_received)
for event_type, count in counts.most_common():
    print(f"  {event_type:30s}  {count}")

## 3b. Events from background work

Once a workflow has [companion threads](19_companion_threads.ipynb) or
[evaluate-while-waiting edges](20_waiting_conditions.ipynb), the stream carries more than one
thread's worth of events — and three lifecycle events that used to coincide diverge:

| Type | Emitted when |
|---|---|
| `end_workflow_iteration` | One turn's accounting is done. **Companions may still be running.** |
| `workflow_ready_for_input` | The main thread waits for the user while companions run. Sent only then. **Send the next message.** |
| `end_workflow` | Every thread has ended, companions included. |

`event.is_ready_for_input()` is the hand-over to the user in every workflow: on `workflow_ready_for_input`
while companions run, and on the turn's `end_workflow_iteration` otherwise. Breaking on
`end_workflow_iteration` yourself races once companions exist.

Background and retry events:

| Type | Meaning |
|---|---|
| `companion_step_boundary` | One companion self-loop step finished. |
| `waiting_evaluation_boundary` | A parked thread's waiting-edges were evaluated. |
| `waiting_condition_matched` | A waiting edge fired with **no user message**. |
| `self_loop_delay` / `self_loop_exhausted` | A bounded self-loop is retrying / gave up. |
| `node_expired` | An in-flight execution was terminated by `expiry_time`. |
| `guardrail_escalation_edge` | The guardrail-strikes threshold escalated the run. |

In [ ]:
# Every event carries `thread_reference_id`: "0" for the main thread, or a companion's
# configured thread_id. It is a COMPUTED field, derived from the internal thread id rather
# than stored, so it is present even on events written before companions existed.
for event in events_received[:5]:
    print(f"{event.type:28s} thread={event.thread_reference_id!r}")

print("\nthreads seen in this run:", {e.thread_reference_id for e in events_received})

### Event-specific fields are top-level, not `event.data`

`RunEvent` is `extra="allow"`, so fields the server sends that the model does not declare are
preserved as ordinary attributes. `event.data` is a legacy slot the server never populates —
reaching for it gets you `None` every time.

```python
event.outcome, event.total_attempts              # self_loop_exhausted
event.hydrated_condition_expression              # waiting_condition_matched
event.active_companion_thread_ids                # workflow_ready_for_input
```

`active_companion_thread_ids` holds **internal** ids — a companion configured as
`thread_id="labpoll"` appears as `"0_companion_labpoll"`. Match on the suffix, not equality.

## 3c. Close codes and errors

The run socket closes with one code per outcome. 1000 is the run ending, and iteration simply stops. Every other
code raises a `StreamError` subclass named for it, carrying `code`, `reason` and `server_message`: the full text of
the `workflow_error` frame the server sent before closing, since a close reason holds only 123 bytes.

| Close code | Exception | Meaning |
|---|---|---|
| 1008 | `StreamPermissionError` | The caller's role may not run workflows (a viewer) |
| 1011 | `StreamServerError` | The server failed while handling the run |
| 4000 | `StreamRejectedError` | The opening frame or the workflow's configuration was refused |
| 4001, 4002, 4005 | `UnrunnableWorkflowError` | The stored workflow cannot run |
| 4003 | `ConcurrencyLimitError` | Too many runs at once |
| 4006 | `InvalidStreamInputError` | The frame carried `initial_state`; use a `rerun_token` |
| 4007 | `RerunTokenError` | The re-run token expired or was minted for something else |
| 4009 | `RunChangedElsewhereError` | The run was paused, resumed or cancelled elsewhere |
| 4010 | `TakeBackRefusedError` | The run could not be taken back |
| 4030 | `TakeBackForbiddenError` | A take-back by a role below developer |
| 4040 | `StreamNotFoundError` | The workflow or run does not exist |
| 4400 | `ResumeInputRejectedError` | The frame carried a resume input |

A code not in the table raises a plain `StreamError`. Every one subclasses `StreamError`, so `except StreamError`
catches them all. When each is sent: [`../docs/streaming.md`](../docs/streaming.md#close-codes-and-errors).

In [ ]:
from interactly import RerunTokenError, StreamError


def describe(exc: StreamError) -> str:
    return f"{exc.code} -> {type(exc).__name__}: {(exc.server_message or exc.reason or '')[:120]}"


# A re-run token the server never minted: close 4007.
try:
    async with client.runs.stream(workflow_id=WORKFLOW_ID, rerun_token="not-a-real-token") as bad:
        async for _ in bad:
            pass
except RerunTokenError as exc:
    print(describe(exc))

# Taking back a run: 4040 for one that does not exist, or 4010 on a server with durable socket runs off.
try:
    async with client.runs.stream(workflow_id=WORKFLOW_ID, workflow_run_id="0" * 24) as gone:
        async for _ in gone:
            pass
except StreamError as exc:
    print(describe(exc))

## 4. Contrast: the ordinary HTTP path (`execute()`)

When you only need the *final* result — not a live feed — use `client.runs.execute()`. That
is a normal **HTTP POST** (request in, one response out); no persistent socket. It returns an
`InteractiveRunResponse`; echo its `run_id` on each subsequent turn until `is_completed`.

Rule of thumb: **`stream()` (WebSocket)** when you want events as they happen; **`execute()`
(HTTP)** when you just want the outcome of a turn.

In [ ]:
from interactly.types.runs.interactive_run import InteractiveRunResponse

result: InteractiveRunResponse = await client.runs.execute(WORKFLOW_ID)

print(f"Run id            : {result.run_id}")
print(f"Status            : {result.status}")
print(f"Turn number       : {result.turn_number}")
print(f"Waiting for input : {result.is_waiting_for_input}")
print(f"Completed         : {result.is_completed}")

## Cleanup

Delete the demo workflows and close the client.

In [ ]:
await client.workflows.delete(WORKFLOW_ID)
await client.workflows.delete(CHAT_WORKFLOW_ID)
await client.close()
print("Cleaned up workflows", WORKFLOW_ID, CHAT_WORKFLOW_ID)

## See also

- Guide: [`../docs/streaming.md`](../docs/streaming.md) — sending turns, stopping, taking a run back, close codes
- [`12_runtime_handles.ipynb`](12_runtime_handles.ipynb) — multi-turn chat via a runtime handle
- [`14_pagination_and_filtering.ipynb`](14_pagination_and_filtering.ipynb) — listing past runs and their events